# Setup

Install DaliScope following the root README before launching this notebook and select the **Python (DaliScope)** kernel. If packages were updated in this session, restart the kernel once. Run cells in order. The cells below locate the repository and set the working directory to `notebooks` for the relative data paths.

In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd().resolve()
if not (ROOT / "daliscope").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "daliscope").is_dir():
    raise RuntimeError("Start in the DaliScope root or notebooks directory")
os.chdir(ROOT / "notebooks")
import daliscope
print("DaliScope", daliscope.__version__)

In [ ]:
import pandas as pd
from IPython.display import display

import daliscope
import daliscope.core.project
import daliscope.optics.histograms
import daliscope.analysis.data_preview
import daliscope.analysis.occupancy
import daliscope.analysis.structal_similarity # for cluster evaluation
import daliscope.widgets.pairwise_py3Dmol
import daliscope.optics.modules_sunburst
import daliscope.analysis.communities
import daliscope.analysis.pfam_stats
import daliscope.optics.py3Dmol_viewer
import daliscope.optics.domnet_viewer
import daliscope.analysis.CH_cluster

from daliscope.analysis.data_preview import show_heading

# A multidomain Query: from architecture to sequence signatures

In this worked example, we use a multidomain Query protein to explore how structural and sequence features can be traced from an individual protein into a broader population.

We begin by examining the Query's domain architecture and structural relationships, then use sequence-level analyses to identify conserved features within selected domains. These observations lead to a discrete sequence signature, which we use to search the raw dataset and determine where the corresponding feature occurs across the population. Finally, we summarize the functional descriptions associated with the resulting modules.

The example therefore follows a progression from **domain architecture and structural context, through sequence conservation, to population-wide pattern search and functional interpretation**, illustrating how the different DaliScope analyses can be combined into a single investigation.


<hr style="border: 2px solid #ccc;">

# <span style="color: #2b5c8f;">Explore the Dali search results</span>

<div style="color: #555555; font-size: 1.1em; font-style: italic; margin-top: -8px; margin-bottom: 12px;">
    Structural similarity at a glance
</div>

---

Once the raw Dali data are loaded, DaliScope presents the search results as a scatterplot of **Z-score versus query coverage**. Together, these provide a first view of the quality and extent of each structural match.

The two measures are generally correlated. The Dali Z-score summarizes structural similarity based on the correspondence of intramolecular distances: for a given number of aligned residues, a more precise structural match produces a higher Z-score.

The plot uses two complementary colour schemes:

* **Sequence identity** indicates how the structural matches relate to what could be found by sequence-based searches: from clear sequence homologs, through the *twilight zone* where structural alignment becomes particularly informative, to cases where structural similarity is difficult to detect from sequence alone.
* **Z-score regimes** provide a heuristic indication of the strength of the structural match. These regimes work well for typical cases, but their interpretation depends on the query structure: optimal thresholds can vary with protein size and shape.

As a first step, this view gives us a map of the structural neighbourhood of the Query and helps identify which hits warrant closer inspection.


In [ ]:
# Load data
# The data pack must be uploaded to the Notebook host. Default data files are in DaliScope/notebooks/data.
pack_path = 'data/ZN_full.tar.gz'

# 1. Initialize & Preview Top Hits
project, FULL_DF, FULL_VIEW = daliscope.analysis.data_preview.load_and_inspect_project(pack_path)

# 2. Query metadata
show_heading("Query information")
tmp = list(project._query_meta.iloc[0])
puu_domains = tmp[5]
print(f"""
Identifier: {tmp[0]}
Length: {tmp[1]}
Description: {tmp[4]}
PUU domains: {puu_domains}
""")

# 3. Plot Landscape
fig = daliscope.analysis.data_preview.plot_dali_hits_with_boundaries(
    FULL_DF, 
    daliscope.analysis.data_preview.build_and_register_subsets(project, FULL_DF, FULL_VIEW)
)

# 4. Project structural alignments onto Query positions
project.viz.plot_msa(FULL_VIEW, 0, project.query_length, plot_type='heatmap', data_col='dssp_pileup')

>The plot shows a typical mix of sequence identity regimes: homologs have the highest Z-scores, followed by twilight hits. The large peak with dark hits can contain genuine biologically interesting hits, and noise. >DaliScope automatically created two subsets wihch bisect the plane at valleys in the density distribution. Sometimes - but far from always - the subsets correspond to families or superfamilies. These automatically generated subsets are called FILTERED_0 and FILTERED_1.
>
>The structural alignment shows mutliple mutually exlusive blocks - the Query protein consists of multiple domains, some with more hits than others. Refined superimposition and accurate annotation need to be applied on a per-domain basis.

## Structural Coherence & Weighted Alignment

Dali alignments optimize intramolecular distance similarity rather than spatial coordinates. Consequently, distant $\text{C}_\alpha$ atoms at similar radial distances can inflate global RMSD values and disrupt rigid-body superimpositions.

To restore spatial coherence within specified domains, we apply a two-step weighted superimposition:
1. **Domain Clipping:** Assigns zero weight to off-domain aligned positions.
2. **Occupancy Weighting:** Downweights highly flexible regions (e.g., irregular loops) to prevent them from distorting core alignments.

In the secondary-structure-colored alignment plot (blue $\alpha$-helices, red $\beta$-strands, green loops), rigid domains appear as distinct structural blocks. Using this visual reference, we define five contiguous domains: `1–149`, `150–214`, `840–920`, `921–1100`, and `1343–1722`.

Each domain yields two structural sets:
* **`dom_{i}`**: Contains domain-clipped alignments with updated metrics.
* **`dom_{i}_FOLD`**: Subsets targets covering $\ge 80\%$ of the domain core.

In [ ]:
# Prepare PDB structure string
pdb_string = daliscope.optics.py3Dmol_viewer.xyz_to_ca_pdb(project.query_ca_coords, project.query_sequence)

custom_domains = "1-149, 150-214, 840-920, 921-1100, 1343-1722"

# Initialize and render interactive viewer
viewer = daliscope.optics.domnet_viewer.DomNetViewer(
    pdb_text=pdb_string,
    domain_string=custom_domains if custom_domains else puu_domains,
    normal_radius=0.65,
    highlight_radius=0.60,
)

print("""
Domain Syntax example: "1-100, 101-300_400-500, 301-399" means 
    Domain 1: 1–100; 
    Domain 2 [discontiguous]: 101–300 + 400–500; 
    Domain 3: 301–399
    
IF the viewer throws a JavaScript error, refresh your browser (CTRL-R) and rerun this code cell. 
If the canvas is blank, re-run this cell

You can play around editing and visualizing domain boundaries, though this worked example will use 
the initial domain boundaries for downstream analyses.
""")

viewer.display()

In [ ]:
# 2. Register clipped parent domain. Updates superimpositions and alignment metrics (except Z-score which stays global)
daliscope.mechanics.metrics.register_multiple_domains(
    domain_string=custom_domains, 
    project=project, 
    custom_name="dom"
)

# Process each registered domain view
view_list = []
for i in [1,2,3,4,5]: 
    view_list.append(f"dom_{i}")

for view_name in view_list:
    daliscope.analysis.occupancy.create_fold_subset(project, view_name, show_plot=True, cutoff=0.5)

show_heading("Which parts of the protein correspond to distinct structural folds?")
shared_matrix = daliscope.analysis.occupancy.get_fold_matrix(project, view_list)

# Render heatmap
daliscope.analysis.occupancy.plot_shared_matrix_heatmap(shared_matrix, figsize=(3,2))

> dom_3_FOLD always co-occurs with dom_4_FOLD, and dom_4 always co-occurs with dom_5_FOLD.
> 
> dom_1_FOLD and dom_2_FOLD are evolutionarily mobile modules: they recombine with target proteins that share no other domains with our Query protein.

# What do existing domain classifications tell us?
Here, we show the Pfam family and superfamily (clan) assignments of the domain-fold populations. The table links family and clan identifiers to InterPro documentation. Domains 3-5 are found in the CUPIN superfamily. Domains 1 and 2 have an SH3-like fold.

In [ ]:
# 1. Table view
for i in [1,2,3,4,5]:
    view_name = f"dom_{i}_FOLD"
    show_heading(view_name)
    daliscope.analysis.pfam_stats.pfam_counts(view_name, project, k=10, print_tables = True)

# Play with the structures
The Query structure is blue, the target is red. Thick lines and dashed lines show the raw Dali alignment, but the least-quares superimposition is based on the clipped and weighted equivalences. 

Note that Dali performs a local structural alignment, finding matches between small domains embedded in large multidomain proteins.

In [ ]:
# omteractive py3Dmol viewer
daliscope.widgets.pairwise_py3Dmol.generate_superimposition_view(
    target_ix=77,
    df=project.views['dom_2_FOLD'],
    project=project,
    q_ca_coords=project.query_ca_coords,
    target_ca_coords=project.coords,
)

## How do domain architectures vary across the target population?

Domain architecture cartoons provide a compact overview of domain composition and organization across the target population.

* **Dashed red boxes** mark the boundaries of the structural match.
* **Empty boxes** represent missing Pfam annotations.
* **`dom_3_FOLD` and `dom_4_FOLD`** show distinct association with CUPIN domains that are not currently captured by Pfam.* **Partially occupied boxes** in `dom_5_FOLD` indicate that the structurally conserved core extends beyond Pfam's sequence profile model.

In [ ]:
# Domain architecture cartoons
project.viz.architecture('dom_1_FOLD', renderer='matplotlib', pfam_col='clan_domains')

In [ ]:
# Domain architecture cartoons
project.viz.architecture('dom_2_FOLD', renderer='matplotlib', pfam_col='clan_domains')

In [ ]:
# Domain architecture cartoons
project.viz.architecture('dom_3_FOLD', renderer='matplotlib', pfam_col='clan_domains')

In [ ]:
# Domain architecture cartoons
project.viz.architecture('dom_4_FOLD', renderer='matplotlib', pfam_col='clan_domains')

In [ ]:
# Domain architecture cartoons
project.viz.architecture('dom_5_FOLD', renderer='matplotlib', pfam_col='clan_domains')

## Where do the domain populations fall in the scatterplot?

The five domains preserve the original Z-scores calculated from the full-length protein comparisons. As a result, hits matching smaller individual domains cluster at lower Z-scores compared to those spanning larger or multiple domains. Notably, the CUPIN superfamily (CL0029) forms a distinct population corresponding to the `dom_3_FOLD`, `dom_4_FOLD` and `dom_5_FOLD` clusters identified above.

In [ ]:
# 3. Scatterplot colored by Pfam categories. Pool more distant Pfam categories as 'Other'
def _scatterplot(view_name):
    show_heading(view_name)
    df = project.views[view_name][['alignment_length', 'z_score', 'clan', 'pfam']].copy()
    df["pfam_label"] = daliscope.analysis.pfam_stats.top_category_label(df, "pfam", k=10)
    df["clan_label"] = daliscope.analysis.pfam_stats.top_category_label(df, "clan", k=10)
    display(daliscope.optics.histograms.plot_scatter(df, x='z_score', y='alignment_length', color='clan_label', engine='matplot',))

for i in [1,2,3,4,5]: 
    view_name = f'dom_{i}_FOLD'
    _scatterplot(view_name)

# What sequence and functional features characterize the domain populations?

Sequence logos and functional annotation summaries highlight key functional motifs across the five structural domains.

## Sequence Logos

* **Domains 1 & 2 (SH3-like):** Low information content overall, reflecting broad sequence divergence across the fold.
* **Domains 3 & 4:** Characterized by highly conserved cysteines across the structural alignment.
* **Domain 5:** Features a distinct $\text{HxD}\dots\text{H}$ active-site motif.

In [ ]:
## Sequence Logos
project.viz.plot_msa("dom_1_FOLD", 1, 149)
project.viz.plot_msa("dom_2_FOLD", 150, 214)
project.viz.plot_msa("dom_3_FOLD", 840, 920)
project.viz.plot_msa("dom_4_FOLD", 921, 1100)
project.viz.plot_msa("dom_5_FOLD", 1500, 1660)

# Is there a conserved active/binding site in your structure?
*Map the conserved residues in the 3-D structure*

Above, sequence logos shows conserved cysteines and histidines, and a putative catalytic signature. Here, we search for potential metal binding sites by checking side chain geometry in the Query structure. Then, we highlight the identified sites in the 3D structure. 

In [ ]:
# Check metal binding sites in Query. Requires full coordinates.
results = daliscope.analysis.CH_cluster.evaluate_cys_his_sites(project.query_pdb_str)
daliscope.analysis.CH_cluster.print_report(results)

In [ ]:
# Positions identified as metal-binding or active site candidates
important_residues = [1523, 1525, 1652, 867, 877, 880, 900, 962, 1075, 1087, 1089, 994, 997, 1016, 1019, 1008, 1011, 1045, 1051]

# Highlight candidate metal binding residues and active site
prop = {} 
for i in important_residues[0:3]: prop[i] = 5   # domain 5 active site
for i in important_residues[3:7]: prop[i] = 4   # metal binding candidate
for i in important_residues[7:11]: prop[i] = 3  # metal binding candidate
for i in important_residues[11:15]: prop[i] = 2 # metal binding candidate
for i in important_residues[15:19]: prop[i] = 1  # metal binding candidate

daliscope.optics.py3Dmol_viewer.show_structure(
                    project.query_pdb_str,
                    important_residues,
                    property_dict=prop,
                    width=800,
                    height=600,
                    backbone_radius=0.18,
                    ca_radius=0.28,
                    highlight_scale=1.8)

>The AlphaFold model has built in four potential metal binding sites with C4 or C3H composition. The side chains are oriented for tetrahedral coordination of a metal ion. The CxxC spacing of cysteines along the sequence is characteristic of a "cysteine knuckle". The geometrical check confirmed bond lengths in valid range. The active site (red) has two histidines and an aspartate.

# Which proteins have the active site's Signature Motif?

We search the full data set for matches to the HxD...H signature motif. This motif would be totally uninformative if applied to a sequence database - the distinction here is that our search space is anchored to specific positions in the structural alignment. Thus, what we identify rae targets that have a similar fold to the query **and** carry the signature motif.

In [ ]:
import daliscope.optics.histograms

# define search population. We use a domain population for accurate Pfam annotations.
view_name = 'dom_5_FOLD'
parent_df = project.views[view_name]

daliscope.optics.histograms.set_motif(parent_df, [1523, 1525, 1652], k=4, verbose=True)

# Define all 3 patterns in the list
patterns = [
    [(1523, ["H"]), (1525, ["D"]), (1652, ["H"])],
    [(1523, ["H"]), (1525, ["D", "H"]), (1652, ["H"])],
    [(1523, ["H"]), (1652, ["H"])],
]

view = project.views[view_name]

# Loop over each pattern one by one
for i, pattern in enumerate(patterns, start=1):
    show_heading(f"Pattern {pattern}")
    # Pass a single pattern wrapped in a list: [pattern]
    hits = daliscope.optics.histograms.find_pattern_targets(view, [pattern])
    
    print(f"\n[{view_name}] Pattern {i}: {len(hits)} matches")
    print(f"Pattern definition: {pattern}\n")

    for category in ['clan', 'pfam']:
        # plot penetrance per category for this pattern
        daliscope.optics.histograms.draw_diverging_percentage_plot(
            view,
            hits=hits,
            grouping_col=category,
            yscale='linear',
        )

>The HxD...H motif is found in 697 CUPIN domains and 15 unassigned domains. Within the CUPIN superfamily, four families show almost 100% penetrance while three smaller families show partial penetrance.
>
>Though HxD...H was suggested by the sequence logos, enumeration of residues occurring at the important positions shows that the second position is wobbly. If only the first and last histidine are required, we recall 96% of the known CUPIN superfamily members.

In [ ]:
# 1. Get hits for the specified pattern
hits = daliscope.optics.histograms.find_pattern_targets(view, [patterns[1]])

# 2. Filter for hits where pfam is 'Unassigned'
df = project.views[view_name]
unassigned_hits_df = df[
    (df['target_id'].isin(hits)) & 
    (df['pfam'] == 'Unassigned')
]

# 3. Verify sequence context
tmp = list(unassigned_hits_df['sequ_pileup'])
for row in tmp:
    print(row[1500:1540],' ... ',row[1620:1660])

# 4. Print target_id and description
print(f"\nFound {len(unassigned_hits_df)} hits with 'Unassigned' Pfam:\n")
with pd.option_context('display.max_colwidth', None):
    display(unassigned_hits_df[['target_id', 'z_score', 'description']]) #.to_string(index=False))

>The unassigned hits include dioxygenase and JmjC domain annotations, which are members of the CUPIN superfamily. Half of then appear wholly uncharacterized.

## Summarizing functional descriptions per module (t.b.i)
For each module, we select representative functional descriptions using a TF-IDF-based approach.

This section is a planned extension and is not executed in this notebook.

# Conclusion

In this worked example, **DaliScope** successfully uncovered the full structural architecture of a query protein whose sequence-based Pfam annotations were sparse and incomplete:

* **Unmapped Domain Boundaries:** We delineated all five recurrent structural domains comprising the query protein, establishing precise boundaries that sequence profiles failed to resolve.
* **Identification of Tudor & Shadow Domains:** We identified two **SH3-like (Tudor)** domains and two previously uncharacterized **CUPIN shadow domains**.
* **Structural Co-occurrence & Function:** We demonstrated that these shadow domains reliably co-occur alongside the canonical CUPIN domain and harbor **four distinct metal-binding sites**, pointing to a potential regulatory mechanism for CUPIN activity.

Ultimately, this analysis illustrates the power of **3D structural alignment in DaliScope** to go beyond primary sequence matching—filling critical gaps in Pfam coverage, revealing hidden regulatory modules, and expanding our understanding of multidomain protein evolution.

In [ ]:
# This is how the Query is annotated by Pfam
project.viz.architecture(FULL_VIEW, renderer='matplotlib', pfam_col='pfam_domains', max_targets=1)